# Instruction Finetuning on Gemma-2b


### Installation

In [ ]:
!pip install transformers==4.51.3  bitsandbytes==0.45.3 datasets peft accelerate trl safetensors

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## **Inference on Gemma Base Model**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

token="" ## Use Your Token for Gated Models

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", token=token)
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b",token=token, quantization_config=quantization_config)

In [ ]:
prompt = """State Section 1 of the Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

print(tokenizer.decode(outputs[0]))

In [ ]:
prompt = """
The Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016
Chapter I
Preliminary
1. Short title, extent and commencement.-
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

print(tokenizer.decode(outputs[0]))

In [ ]:
prompt = """
A fever is a temporary increase in body temperature, usually
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

print(tokenizer.decode(outputs[0]))

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## **Instruction Finetuning**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
use_flash_attention = True

model_id = "google/gemma-2b" 
 
 
# token ="Your HF Token"
 
 
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True
)
 
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    use_cache=False,
    device_map="auto",
    # token=token,
)
 
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## **PEFT Configration**

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
 
# LoRA config based on QLoRA paper
peft_config = LoraConfig(
        lora_alpha=64,
        lora_dropout=0,
        r=128,
        bias="none",
        task_type="CAUSAL_LM",
)
 
 
# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [ ]:
import matplotlib.pyplot as plt

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

plt.figure()
plt.pie(
    [trainable, frozen],
    labels=["Trainable (LoRA)", "Frozen (Base Model)"],
    autopct="%1.2f%%"
)
plt.title("PEFT Parameter Distribution")
plt.show()

<a name="Data"></a>
### Data Prep

In [ ]:
from datasets import load_dataset
from random import randrange

dataset = load_dataset("mratanusarkar/Indian-Laws", split="train")

total = len(dataset)
one_percent = int(total * 0.05)

dataset = dataset.select(range(0, one_percent)) ## Only using 1% of total dataset for faster training

print("Dataset Lenghth:", len(dataset))

In [ ]:
print(dataset[0])

In [ ]:
def format_instruction(sample):
	return f"""### Instruction:
You are an Indian Law Expert. Write a response that appropriately completes the request.
 
### Input:
State Section {sample["section"]} of {sample["act_title"]}
 
### Response:
{sample['law']}
"""

In [ ]:
from random import randrange
 
print(format_instruction(dataset[randrange(len(dataset))]))

We only use 5% of the dataset to speed things up! Use more for longer runs!

## **SFT Configration**

In [ ]:
from trl import SFTConfig
max_seq_length = 2048 

 
args = SFTConfig(
    output_dir="gemma-ft-law",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    max_length=max_seq_length,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=25,
    packing=True,
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=True,
    tf32=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    disable_tqdm=False 
)

## **Model Finetuting**

In [ ]:
from trl import SFTTrainer
 
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=format_instruction,
    args=args,
)

In [ ]:
# train
trainer.train() 

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## **Merging and Saving finetuned model**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# ---- Config ----
base_model_id = "google/gemma-2b" # base model
adapter_path = "gemma-ft-law/checkpoint-555/"  # your fine-tuned adapter path
output_path = "Merged-IF-Gemma-2b-Indina-Law"  # Merged output

# ---- Load base model in full precision (no quant) ----
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ---- Load LoRA adapter and merge ----
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

# ---- Save merged model ----
model.save_pretrained(output_path)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained(output_path)

print(f"Merged model saved at {output_path}")

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## **Perplexity Score Measurement**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_name = "google/gemma-2b" 

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quantization_config
)

In [ ]:
alpaca_prompt = """You are an Indian Law Expert. Write a response that appropriately completes the request.

### Instruction:
State Section {} of {}

### Response:
{}"""


EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(batch):
    texts = []
    for act_title, section, law in zip(batch["act_title"], batch["section"], batch["law"]):
        text = alpaca_prompt.format(section, act_title, law) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

In [ ]:
from datasets import load_dataset

dataset = load_dataset("mratanusarkar/Indian-Laws", split="train")

total = len(dataset)
subset_size = int(total * 0.005)
dataset = dataset.select(range(subset_size))

print("Eval Dataset Length: ", len(dataset))

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
texts = formatted_dataset["text"]

In [ ]:
import math

def compute_perplexity(texts, model, tokenizer, max_length=512):
    losses = []

    for text in texts:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            outputs = model(
                **enc,
                labels=enc["input_ids"]
            )
            losses.append(outputs.loss.item())

    mean_loss = sum(losses) / len(losses)
    return math.exp(mean_loss)

In [ ]:
ppl = compute_perplexity(texts, model, tokenizer)
print("Base Gemma Perplexity Score:", ppl)

In [ ]:
model_name = "Merged-IF-Gemma-2b-Indina-Law" 

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quantization_config
)

In [ ]:
ppl = compute_perplexity(texts, model, tokenizer)
print("Instruction Finetuned Gemma Perplexity Score:", ppl)

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## **Inference on Gemma-2b Instruction Finetuned Model**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

token="" ## Use Your Token for Gated Models

tokenizer = AutoTokenizer.from_pretrained("Merged-IF-Gemma-2b-Indina-Law")
model = AutoModelForCausalLM.from_pretrained("Merged-IF-Gemma-2b-Indina-Law", quantization_config=quantization_config)

In [ ]:
prompt = """You are an Indian Law Expert. Write a response that appropriately completes the request.

### Instruction:
State Section 1 of the Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016

### Rersponse:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

print(tokenizer.decode(outputs[0]))

In [ ]:
prompt = """You are an Indian Law Expert. Write a response that appropriately completes the request.

### Instruction:
State Section 11 of the Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016

### Rersponse:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
)

print(tokenizer.decode(outputs[0]))